**Aprendizaje no supervisado**

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN, KMeans
import pandas as pd
from matplotlib.ticker import PercentFormatter

In [ ]:
Funciones a utilizar

In [2]:
# Radio medio aproximado de la Tierra en kilómetros.
radio_tierra_km = 6371.0088

# Función para implementar DBSCAN geográfico

def dbscan_geo(coordenadas_radianes, eps_km, min_samples=5):
    """
    Aplica DBSCAN utilizando distancia haversine.

    Parámetros
    ----------
    coordenadas_radianes:
        Matriz de coordenadas [latitud, longitud] en radianes.

    eps_km:
        Radio máximo de vecindad expresado en kilómetros.

    min_samples:
        Número mínimo de observaciones en una vecindad
        para identificar un punto central.

    Retorna
    -------
    etiquetas:
        Cluster asignado a cada observación.

        0, 1, 2, ... : identificadores de clusters.
        -1           : observaciones clasificadas como ruido.
    """

    # Convertimos la distancia de kilómetros a radianes.
    # Ejemplo: 1 km / 6371.0088 km ≈ 0.000157 radianes.

    eps_rad = eps_km / radio_tierra_km

    modelo = DBSCAN(
        eps=eps_rad,
        min_samples=min_samples,
        metric="haversine",
        algorithm="ball_tree"
    )

    # fit_predict ajusta DBSCAN y devuelve directamente
    # el cluster asignado a cada observación.
    etiquetas = modelo.fit_predict(coordenadas_radianes)

    return etiquetas

**Datos**

Importamos la base de datos

In [3]:
url = "https://raw.githubusercontent.com/ElenaVillano/databases/refs/heads/main/db_falsas_alarmas.csv"
df = pd.read_csv(url)
df.head(1)

,folio,fecha_hora_creacion,fecha_hora_cierre,tipo_incidente_c4,incidente_c4,alcaldia_inicio,codigo_cierre,clas_con_f_alarma,tipo_entrada,alcaldia_cierre,alcaldia_catalogo,colonia_catalogo,longitud,latitud
0,C5/20220923/04016,2022-09-23 18:38:34,2022-09-23 22:22:27,Accidente,Choque sin lesionados,VENUSTIANO CARRANZA,A,EMERGENCIA,LLAMADA DEL 911,VENUSTIANO CARRANZA,Venustiano Carranza,20 De Noviembre,-99.108352,19.44392


Quitamos algunos puntos fuera del polígono de la CDMX 

In [4]:
df['cordenada_fuera_cdmx'] = ~(df['latitud'].between(19.0482, 19.5928) &df['longitud'].between(-99.3649, -98.9403))
df = df[df['cordenada_fuera_cdmx']==False]

In [5]:
df.columns

Index(['folio', 'fecha_hora_creacion', 'fecha_hora_cierre',
       'tipo_incidente_c4', 'incidente_c4', 'alcaldia_inicio', 'codigo_cierre',
       'clas_con_f_alarma', 'tipo_entrada', 'alcaldia_cierre',
       'alcaldia_catalogo', 'colonia_catalogo', 'longitud', 'latitud',
       'cordenada_fuera_cdmx'],
      dtype='str')

In [6]:
# Ver el tipo de valor
df[['longitud', 'latitud']].dtypes

longitud    float64
latitud     float64
dtype: object

In [7]:
# Ver los datos 
df[['longitud', 'latitud']].head(1)

,longitud,latitud
0,-99.108352,19.44392


Ver los valores faltantes

In [8]:
#No hay valores faltantes

df[['latitud', 'longitud']].isna().sum()

latitud     0
longitud    0
dtype: int64

Tomamos una muestra 

El conjunto de información es muy grande y el algoritmo puede tardar en correr mucho o no encontrar las agrupaciones correctas

In [9]:
df_muestra = df.sample(
    frac=0.05,
    random_state=96
).copy()

print(df_muestra.shape)

(6213, 15)


In [10]:
## Seleciconamos solamente las columnas de longitud y latitud
coordenadas = df_muestra[['longitud','latitud']] 